<center><h1>Vora_Raj_HW3</h1></center>
<br>
<br>

Name: Raj Vora
<br>
Github Username: RajVora622
<br>
USC ID: 9094527849

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.float_format", lambda value: f"{value:.4f}")

Get the AReM Data Set

In [2]:
data_dir = Path("Homework 3, 4 Data/AReM")

series_names = [
    "avg_rss12", "var_rss12", "avg_rss13",
    "var_rss13", "avg_rss23", "var_rss23",
]

activities = [
    "bending1", "bending2", "cycling", "lying",
    "sitting", "standing", "walking",
]

instances = {}
instance_records = []

for activity in activities:

    paths = sorted(
        (data_dir / activity).glob("dataset*.csv"),
        key=lambda path: int(path.stem.removeprefix("dataset")),
    )

    for path in paths:
        
        recording = pd.read_csv(
            path, comment="#", header=None, sep=r"[\s,]+",
            engine="python", usecols=range(7),
        )

        recording.columns = ["time"] + series_names
        recording = recording.apply(pd.to_numeric, errors="raise")
        
        instance_id = len(instances) + 1
        instances[instance_id] = recording[series_names]
        instance_records.append({
            "instance": instance_id,
            "activity": activity,
            "dataset": int(path.stem.removeprefix("dataset")),
            "file": path.relative_to(data_dir).as_posix(),
            "measurements": len(recording),
        })

instance_metadata = pd.DataFrame(instance_records).set_index("instance")

activity_summary = instance_metadata.groupby("activity").agg(
    instances=("dataset", "size"),
    minimum_measurements=("measurements", "min"),
    maximum_measurements=("measurements", "max"),
).reindex(activities)

display(activity_summary)
display(pd.DataFrame({
    "time series": range(1, 7),
    "column": series_names,
}))

print(f"Total recording instances: {len(instances)}")
print(f"Total observed measurements: {instance_metadata['measurements'].sum()}")

,instances,minimum_measurements,maximum_measurements
activity,,,
bending1,7,480,480
bending2,6,480,480
cycling,15,480,480
lying,15,480,480
sitting,15,479,480
standing,15,480,480
walking,15,480,480


,time series,column
0,1,avg_rss12
1,2,var_rss12
2,3,avg_rss13
3,4,var_rss13
4,5,avg_rss23
5,6,var_rss23


Total recording instances: 88
Total observed measurements: 42239


### (b) Test and Train Data

In [3]:
test_limits = {activity: 2 if activity.startswith("bending") else 3
               for activity in activities}
instance_metadata["split"] = [
    "test" if row.dataset <= test_limits[row.activity] else "train"
    for row in instance_metadata.itertuples()
]
train_ids = instance_metadata.index[instance_metadata["split"] == "train"]
test_ids = instance_metadata.index[instance_metadata["split"] == "test"]
train_instances = {instance_id: instances[instance_id] for instance_id in train_ids}
test_instances = {instance_id: instances[instance_id] for instance_id in test_ids}

split_summary = pd.crosstab(
    instance_metadata["activity"], instance_metadata["split"]
).reindex(index=activities, columns=["train", "test"])
split_summary.loc["Total"] = split_summary.sum()
display(split_summary)
assert len(train_instances) == 69
assert len(test_instances) == 19
assert set(train_ids).isdisjoint(test_ids)
assert set(train_ids) | set(test_ids) == set(instances)

split,train,test
activity,,
bending1,5,2
bending2,4,2
cycling,12,3
lying,12,3
sitting,12,3
standing,12,3
walking,12,3
Total,69,19


Datasets 1 and 2 from each bending folder belong to the **test set**. Datasets 1, 2, and 3 from every other activity folder also belong to the test set. All remaining recordings belong to the training set. This gives **69 training instances and 19 test instances**.


### (c) Feature Extraction

#### i. Research

Common time-domain features summarize the values or their changes over time:

| Type | Examples | What they describe |
|:---|:---|:---|
| Location | Mean, median | Typical signal level |
| Extremes | Minimum, maximum, range | Signal bounds and amplitude range |
| Spread | Variance, standard deviation, quartiles, interquartile range | Variation in the signal |
| Distribution shape | Skewness, kurtosis | Asymmetry and tail behavior |
| Magnitude | Root mean square, sum of squared values (energy) | Overall signal magnitude |
| Temporal behavior | Autocorrelation, linear trend, mean absolute change, peak count | Dependence, direction, and changes over time |

The [tsfresh feature documentation](https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html) lists these feature calculations. Time-domain features are computed from the observed values and their ordering; Fourier coefficients are frequency-domain features and are outside this question's scope.

#### ii. Extraction

I extract minimum, maximum, mean, median, sample standard deviation, first quartile, and third quartile from each of the six time series. This gives **7 × 6 = 42 numerical features per instance**. I use the original measurement scale and pandas' default linear interpolation for quartiles.

The within-recording sample standard deviation uses the denominator $m-1$, where $m$ is the number of observed measurements in that recording. The metadata, activity labels, and train/test labels are stored separately from the 42 features.

In [4]:
feature_names = ["min", "max", "mean", "median", "std", "q1", "q3"]
feature_records = []

for instance_id, recording in instances.items():
    statistics = recording.describe(percentiles=[0.25, 0.5, 0.75])
    record = {}
    for series_id, series_name in enumerate(series_names, start=1):
        values = statistics[series_name].loc[
            ["min", "max", "mean", "50%", "std", "25%", "75%"]
        ]
        for feature_name, value in zip(feature_names, values):
            record[f"{feature_name}{series_id}"] = float(value)
    feature_records.append(record)

feature_data = pd.DataFrame(feature_records, index=instance_metadata.index)
train_features = feature_data.loc[train_ids].copy()
test_features = feature_data.loc[test_ids].copy()

print(f"Combined feature matrix: {feature_data.shape}")
print(f"Training feature matrix: {train_features.shape}")
print(f"Test feature matrix: {test_features.shape}")

for series_id, series_name in enumerate(series_names, start=1):
    columns = [f"{feature_name}{series_id}" for feature_name in feature_names]
    print(f"Time series {series_id}: {series_name} (first five instances)")
    display(feature_data[columns].head())

output_dir = Path("features")
output_dir.mkdir(exist_ok=True)
feature_data.to_csv(output_dir / "arem_features.csv")
instance_metadata.to_csv(output_dir / "instance_metadata.csv")

Combined feature matrix: (88, 42)
Training feature matrix: (69, 42)
Test feature matrix: (19, 42)
Time series 1: avg_rss12 (first five instances)


,min1,max1,mean1,median1,std1,q11,q31
instance,,,,,,,
1,37.2500,45.0000,40.6248,40.5000,1.4770,39.2500,42.0000
2,38.0000,45.6700,42.8128,42.5000,1.4355,42.0000,43.6700
3,35.0000,47.4000,43.9545,44.3300,1.5588,43.0000,45.0000
4,33.0000,47.7500,42.1798,43.5000,3.6707,39.1500,45.0000
5,33.0000,45.7500,41.6781,41.7500,2.2435,41.3300,42.7500


Time series 2: var_rss12 (first five instances)


,min2,max2,mean2,median2,std2,q12,q32
instance,,,,,,,
1,0.0000,1.3000,0.3586,0.4300,0.3226,0.0000,0.5000
2,0.0000,1.2200,0.3724,0.4700,0.2892,0.0000,0.5000
3,0.0000,1.7000,0.4263,0.4700,0.3387,0.0000,0.5000
4,0.0000,3.0000,0.6960,0.5000,0.6309,0.0000,1.1200
5,0.0000,2.8300,0.5360,0.5000,0.4055,0.4300,0.7100


Time series 3: avg_rss13 (first five instances)


...

,min4,max4,mean4,median4,std4,q14,q34
instance,,,,,,,
1,0.0000,7.2300,0.8325,0.5000,0.9657,0.0000,1.1200
2,0.0000,5.7600,0.8764,0.5000,1.0473,0.0000,1.1450
3,0.0000,4.4400,0.4973,0.4300,0.5507,0.0000,0.8300
4,0.0000,5.1500,0.9899,0.8300,0.9537,0.4300,1.3000
5,0.0000,6.4200,0.8419,0.5000,0.9288,0.4300,1.1200


Time series 5: avg_rss23 (first five instances)


,min5,max5,mean5,median5,std5,q15,q35
instance,,,,,,,
1,27.2500,38.2500,34.3113,35.0000,2.1884,33.0000,36.0000
2,27.6700,38.5000,33.0246,33.0000,1.9953,32.0000,34.5000
3,29.0000,38.5000,35.5885,36.0000,1.9996,35.3625,36.5000
4,20.0000,38.6700,33.4939,35.0000,3.8494,30.4575,36.3300
5,23.6700,37.5000,29.8571,30.0000,2.4110,28.4575,31.2500


Time series 6: var_rss23 (first five instances)


,min6,max6,mean6,median6,std6,q16,q36
instance,,,,,,,
1,0.0000,1.9200,0.5706,0.4300,0.5829,0.0000,1.3000
2,0.0000,3.1100,0.5711,0.4300,0.6010,0.0000,1.3000
3,0.0000,1.7900,0.4933,0.4300,0.5135,0.0000,0.9400
4,0.0000,2.1800,0.6135,0.5000,0.5243,0.0000,1.0000
5,0.0000,1.7900,0.3833,0.4300,0.3892,0.0000,0.5000


The previews show the seven extracted features for each time series. For example, `q36` is the third quartile of time series 6, and `std1` is the standard deviation within time series 1 of an instance. The complete **88 × 42** feature matrix is saved as `features/arem_features.csv`; `instance` is its row index. `features/instance_metadata.csv` links each row to its activity, source file, and split.

#### iii. Standard Deviation

Here I estimate the standard deviation of each feature **across the 88 instances** For feature $j$,

$$s_j=\sqrt{\frac{1}{87}\sum_{i=1}^{88}(x_{ij}-\bar{x}_j)^2}.$$

I draw **1,000 bootstrap samples**, each containing 88 instance rows sampled with replacement. Each resample uses the same row indices for all 42 features. I calculate the sample standard deviation of each feature in every resample, then use its 5th and 95th percentiles as the **90% percentile bootstrap confidence interval**. The fixed random seed makes the results reproducible.

In [5]:
n_bootstrap = 1000
rng = np.random.default_rng(42)
feature_values = feature_data.to_numpy()
bootstrap_indices = rng.integers(
    0, len(feature_data), size=(n_bootstrap, len(feature_data))
)
bootstrap_std = feature_values[bootstrap_indices].std(axis=1, ddof=1)
ci_lower, ci_upper = np.percentile(bootstrap_std, [5, 95], axis=0)

standard_deviation_summary = pd.DataFrame({
    "standard deviation": feature_data.std(ddof=1),
    "90% CI lower": ci_lower,
    "90% CI upper": ci_upper,
})
standard_deviation_summary.index.name = "feature"

with pd.option_context("display.max_rows", 50):
    display(standard_deviation_summary)
standard_deviation_summary.to_csv(output_dir / "feature_std_ci.csv")

,standard deviation,90% CI lower,90% CI upper
feature,,,
min1,9.5700,8.2096,10.7872
max1,4.3944,3.3267,5.3222
mean1,5.3357,4.7220,5.8990
median1,5.4401,4.7853,6.0034
std1,1.7722,1.5689,1.9442
q11,6.1536,5.5971,6.6646
q31,5.1389,4.3344,5.8429
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6446,5.3976


The table reports the estimate and confidence interval for all 42 features. For `std1`, for example, the estimate measures how much the within-recording standard deviations vary across recordings. These intervals describe uncertainty in each feature's standard deviation, not uncertainty in its mean. The bootstrap resamples recording instances, so it does not treat the consecutive measurements within each recording as independent observations.

#### iv. Select Features

I select **mean, standard deviation, and maximum** as the three most useful types of time-domain features:

- **Mean** represents the typical RSS level. Different positions and activities can change that level.
- **Standard deviation** represents variation during a recording. It can help distinguish relatively still activities from activities with more movement.
- **Maximum** represents the upper signal level reached during a recording. It adds information about peaks that the mean and standard deviation do not directly describe, although it can be sensitive to outliers.

These describe signal level, variation, and an extreme. 

## 2. ISLR 3.7.4

### (a) Linear Train

The **cubic model's training RSS is less than or equal to the linear model's training RSS**, even when the true relationship is linear. The linear model is a special case of the cubic model with $\beta_2=\beta_3=0$. Least squares minimizes training RSS, so adding these terms cannot make the minimum training RSS larger. In practice, the cubic terms can fit some of the training noise and make its RSS smaller.

### (b) Linear Test

When the true relationship is linear, we would generally expect **linear regression to have lower test RSS**. The linear model already represents the true relationship. Adding quadratic and cubic terms increases estimation variance without reducing the model's approximation bias, so the cubic model can overfit training noise. This is an expectation over samples, rather than a guarantee for every particular test set.

### (c) Not Linear Train

The **cubic model's training RSS is again less than or equal to the linear model's training RSS**. The nesting argument from part (a) holds regardless of the true relationship. The extra terms can fit curvature as well as training noise; equality is possible if the extra terms do not improve the fit.

### (d) Not Linear Testing

There is **not enough information to determine which model has lower test RSS**. The cubic model may reduce bias if its additional terms capture the true nonlinearity, but it also has higher estimation variance. With strong curvature that a cubic model captures well, cubic regression may perform better. With slight curvature, high noise, or a nonlinear shape that a cubic polynomial captures poorly, linear regression may perform better. Knowing only that the relationship is nonlinear does not settle this bias–variance tradeoff.